In [3]:
from sunpy.coordinates.sun import carrington_rotation_time
from astropy.io import fits
import astropy.units as u
from astropy.time import Time
import numpy as np
import sunpy.map
import drms
import subprocess
import sys, os, getopt

In [24]:
50000/(100**3)

0.05

# Cropped Projection Masking

- download affected timestamps (known from cropped LLD data and PHI interface)
- mask ZEROS with BLANK or NP.NAN (try) --> nan works
- reingest into DRMS -> try this without header and see if the old header persists! --> needs header, set_info with all keywords

In [4]:
def get_drms_parameters(inRecs, input_ds):

    #inRecs = "2014.05.12_12:00:00_TAI, 2014.05.13_00:00:00_TAI, 2014.05.13_12:00:00_TAI, 2014.05.14_00:00:00_TAI" # input argument
    
    #nRecs = len(inRecs.split(','))
    #show_info = 'show_info %s["%s"] key="CALVER64,T_REC,QUALITY,FDRADIAL,CARSTRCH,DIFROT_A,DIFROT_B,DIFROT_C,CRVAL1,CRLN_OBS,CAR_ROT,MAPLGMAX,MAPLGMIN,MAPMMAX,I_DREC" -iPA'
    keywords = "DATE,DATE__OBS,TELESCOP,INSTRUME,WAVELNTH,CAMERA,BUNIT,ORIGIN,CONTENT,QUALITY,BLD_VERS,TOTVALS_1,DATAVALS_1,MISSVALS_1,DATAMIN_1,DATAMAX_1,DATAMEDN_1,DATAMEAN_1,DATARMS_1,DATASKEW_1,DATAKURT_1,TOTVALS_2,DATAVALS_2,MISSVALS_2,DATAMIN_2,DATAMAX_2,DATAMEDN_2,DATAMEAN_2,DATARMS_2,DATASKEW_2,DATAKURT_2,TOTVALS_3,DATAVALS_3,MISSVALS_3,DATAMIN_3,DATAMAX_3,DATAMEDN_3,DATAMEAN_3,DATARMS_3,DATASKEW_3,DATAKURT_3,CTYPE1,CTYPE2,CRPIX1,CRPIX2,CRVAL1,CRVAL2,CDELT1,CDELT2,CUNIT1,CUNIT2,CROTA2,CRDER1,CRDER2,CSYSER1,CSYSER2WCSNAME,DSUN_OBS,DSUN_REF,RSUN_REF,CRLN_OBS,CRLT_OBS,HGLN_OBS,CAR_ROT,CAR_ROT2,OBS_VR,OBS_VW,OBS_VN,T_OBS,T_REC,T_REC_epoch,T_REC_step,T_REC_unit,HMI_PREV,HMI_NEXT,CADENCE,DATASIGN,MAPMMAX,MAPBMAX,MAPLGMAX,MAPLGMIN,MAPRMAX,SINBDIVS,LGSHIFT,INTERPO,MCORLEV,MOFFSET,CARSTRCH,DIFROT_A,DIFROT_B,DIFROT_C,CALVER64"
    #keywords = "DATE,DATE__OBS,TELESCOP,INSTRUME,WAVELNTH,CAMERA,BUNIT,ORIGIN,CONTENT,QUALITY,HISTORY,COMMENT,BLD_VERS,TOTVALS_1,DATAVALS_1,MISSVALS_1,DATAMIN_1,DATAMAX_1,DATAMEDN_1,DATAMEAN_1,DATARMS_1,DATASKEW_1,DATAKURT_1,TOTVALS_2,DATAVALS_2,MISSVALS_2,DATAMIN_2,DATAMAX_2,DATAMEDN_2,DATAMEAN_2,DATARMS_2,DATASKEW_2,DATAKURT_2,TOTVALS_3,DATAVALS_3,MISSVALS_3,DATAMIN_3,DATAMAX_3,DATAMEDN_3,DATAMEAN_3,DATARMS_3,DATASKEW_3,DATAKURT_3,CTYPE1,CTYPE2,CRPIX1,CRPIX2,CRVAL1,CRVAL2,CDELT1,CDELT2,CUNIT1,CUNIT2,CROTA2,CRDER1,CRDER2,CSYSER1,CSYSER2WCSNAME,DSUN_OBS,DSUN_REF,RSUN_REF,CRLN_OBS,CRLT_OBS,HGLN_OBS,CAR_ROT,CAR_ROT2,OBS_VR,OBS_VW,OBS_VN,T_OBS,T_REC,T_REC_epoch,T_REC_step,T_REC_unit,HMI_PREV,HMI_NEXT,CADENCE,DATASIGN,MAPMMAX,MAPBMAX,MAPLGMAX,MAPLGMIN,MAPRMAX,SINBDIVS,LGSHIFT,INTERPO,MCORLEV,MOFFSET,CARSTRCH,DIFROT_A,DIFROT_B,DIFROT_C,CALVER64"
    # COMMENT and HISTORY keyword excluded due to the additional \n that breaks the algorithm
    
    show_info = 'show_info %s["%s"] key="%s" -iPA' 
    
    #-P for path and -A for segment

    si_out = subprocess.check_output(show_info %(input_ds,inRecs,keywords) , shell=True)[:-1].decode("utf-8")
    raw = si_out.split('\n')

    formatted = [] 
    drms_param = []

    nRecs = 0
    keys = raw[0].split('\t')

    for line in raw[1:]:  
        formatted = line.split('\t') # [CALVER64, T_REC, QUALITY, FDRADIAL, CARSTRCH, DIFROT_A, DIFROT_B, DIFROT_C, CRVAL1, CRLN_OBS, CAR_ROT, MAPLGMAX, MAPLGMIN, I_DREC]
        dict_tmp = {}
        #print(formatted)
        for i, key in enumerate(keys):
            #print(i, key)
            if formatted[i].strip() == "InvalidKeyname":
                dict_tmp[key] = 0
            else:
                dict_tmp[key] = formatted[i]
   
        drms_param.append(dict_tmp)
        nRecs += 1
    return drms_param

In [5]:
#trec = "2022.02.15_02:39:52_TAI"
data_series_proj = "mps_loeschl.Ml_hiresmap_CR2255"
filename =  "%s_%s.fits"
subdir = "jv2ts_masked/"
path_in = "../output/data/phi/crop/drms/"
path_out = path_in+subdir

if not os.path.isdir(path_out):
    os.mkdir(path_out)

# read nrt trecs from file
trecs = ""
with open(path_in+'trecs_crop.txt', 'r') as trecs_in:
    for line in trecs_in:
        trecs+=line.strip('\n')+',' # build trec string
trecs = trecs[:-1] # remove trailing ,

setinfo_out = open(path_in+'2_jv2ts_masking.sh', 'w')
setinfo_out.write('#!/bin/bash\n')

params = get_drms_parameters(trecs, data_series_proj)

for keywords in params:

    proj = fits.open(keywords['Ml'])

    # filter out zeros from projection of cropped solar disk
    proj[0].data[proj[0].data==0] = np.nan

    # remove DATASUM and CHECKSUM keywords
    proj[0].header = proj[0].header[:-2]

    #save changes
    proj.writeto(path_out+filename%(data_series_proj, keywords['T_REC']), overwrite=True)

    #create keyword string for set_info command
    kwstr = ""
    for key in keywords:
        if key=="query" or key=="Ml":
            continue
        else:
            kwstr+='%s="%s" '%(key, keywords[key])

    # assemble set_info command
    set_info = 'set_info -c ds="%s" %s Ml=./%s%s >> jv2ts_masking.log 2>&1\n'%(data_series_proj, kwstr, subdir, filename%(data_series_proj, keywords['T_REC']))
    
    setinfo_out.write('\necho %s' %set_info)
    setinfo_out.write('\n%s' %set_info)

setinfo_out.close()


KeyboardInterrupt: 

# Visualisation

In [6]:
%matplotlib widget

show_info -iP mps_loeschl.phi_m720s_test[2022.02.15_02:39:52_TAI]

show_info -iP mps_loeschl.Ml_hiresmap_CR2254[2022.02.15_02:39:52_TAI]

In [7]:
trec = "2022.02.15_02:39:52_TAI"

In [8]:
mag  = fits.open("/SUM35/D281475007286530/S00000/magnetogram.fits")[1]
proj = fits.open("/SUM46/D281475007287150/S00000/Ml.fits")
test = fits.open("/SUM46/D281475007287150/S00000/Ml.fits")

In [9]:
plt.figure()
plt.imshow(mag.data, cmap='hmimag', vmin=-1500, vmax=1500, origin="lower")

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [10]:
proj[0].data[proj[0].data==0] = np.nan

In [11]:
plt.figure()
plt.imshow(proj[0].data, vmin=-1500, vmax=1500, cmap='hmimag', origin="lower")

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [12]:
proj[0].header = proj[0].header[:-2]

In [13]:
proj.writeto("../output/data/crop/proj_cropmask.fits", overwrite=True)

In [236]:
#proj3 = fits.open("/SUM36/D281475007289819/S00000/Ml.fits")
#proj3 = fits.open("/SUM47/D281475007291564/S00000/Ml.fits")
proj3 = fits.open("/SUM44/D281475007281040/S00000/Ml.fits")

In [14]:
proj1 = fits.open("/SUM44/D281475007287693/S00000/Ml.fits")
proj2 = fits.open("/SUM34/D281475007289348/S00000/Ml.fits")

In [15]:
plt.figure()
plt.imshow(proj1[0].data, vmin=-1500, vmax=1500, cmap='hmimag', origin="lower")

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [16]:
plt.figure()
plt.imshow(proj2[0].data, vmin=-1500, vmax=1500, cmap='hmimag', origin="lower")

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [23]:
fig, (ax1, ax2, ax3) = plt.subplots(nrows=1, ncols=3)
ax1.imshow(mag.data, vmin=-1500, vmax=1500, cmap='hmimag', origin="lower")
ax2.imshow(proj1[0].data, vmin=-1500, vmax=1500, cmap='hmimag', origin="lower")
ax3.imshow(proj2[0].data, vmin=-1500, vmax=1500, cmap='hmimag', origin="lower")

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [17]:
proj3 = fits.open("/SUM45/D281475007274739/S00000/Ml.fits")
proj4 = fits.open("/SUM44/D281475007286463/S00000/Ml.fits")

In [18]:
plt.figure()
plt.imshow(proj3[0].data, vmin=-1500, vmax=1500, cmap='hmimag', origin="lower")

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [19]:
plt.figure()
plt.imshow(proj4[0].data, vmin=-1500, vmax=1500, cmap='hmimag', origin="lower")

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [20]:
t1 = fits.open(path_out+"mps_loeschl.Ml_hiresmap_CR2255_2022.03.20_20:53:51_TAI.fits")

In [21]:
plt.figure()
plt.imshow(t1[0].data, vmin=-1500, vmax=1500, cmap='hmimag', origin="lower")

Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

In [134]:
# TODO HISTORY KEYWORD ADDS A LINE BREAK AND SPLITS THE KEYWORD VALUES IN RAW[1] AND RAW[2]

In [135]:
# GRAB ALL KEYWORDS AND FEED THEM BACK INTO THE MASKED SEGMENT

In [172]:
params[0]["DATAMIN_1"] == 'nan'

True

In [259]:
def get_drms_parameters(inRecs, input_ds):

    #inRecs = "2014.05.12_12:00:00_TAI, 2014.05.13_00:00:00_TAI, 2014.05.13_12:00:00_TAI, 2014.05.14_00:00:00_TAI" # input argument
    
    #nRecs = len(inRecs.split(','))
    #show_info = 'show_info %s["%s"] key="CALVER64,T_REC,QUALITY,FDRADIAL,CARSTRCH,DIFROT_A,DIFROT_B,DIFROT_C,CRVAL1,CRLN_OBS,CAR_ROT,MAPLGMAX,MAPLGMIN,MAPMMAX,I_DREC" -iPA'
    keywords = "DATE,DATE__OBS,TELESCOP,INSTRUME,WAVELNTH,CAMERA,BUNIT,ORIGIN,CONTENT,QUALITY,COMMENT,BLD_VERS,TOTVALS_1,DATAVALS_1,MISSVALS_1,DATAMIN_1,DATAMAX_1,DATAMEDN_1,DATAMEAN_1,DATARMS_1,DATASKEW_1,DATAKURT_1,TOTVALS_2,DATAVALS_2,MISSVALS_2,DATAMIN_2,DATAMAX_2,DATAMEDN_2,DATAMEAN_2,DATARMS_2,DATASKEW_2,DATAKURT_2,TOTVALS_3,DATAVALS_3,MISSVALS_3,DATAMIN_3,DATAMAX_3,DATAMEDN_3,DATAMEAN_3,DATARMS_3,DATASKEW_3,DATAKURT_3,CTYPE1,CTYPE2,CRPIX1,CRPIX2,CRVAL1,CRVAL2,CDELT1,CDELT2,CUNIT1,CUNIT2,CROTA2,CRDER1,CRDER2,CSYSER1,CSYSER2WCSNAME,DSUN_OBS,DSUN_REF,RSUN_REF,CRLN_OBS,CRLT_OBS,HGLN_OBS,CAR_ROT,CAR_ROT2,OBS_VR,OBS_VW,OBS_VN,T_OBS,T_REC,T_REC_epoch,T_REC_step,T_REC_unit,HMI_PREV,HMI_NEXT,CADENCE,DATASIGN,MAPMMAX,MAPBMAX,MAPLGMAX,MAPLGMIN,MAPRMAX,SINBDIVS,LGSHIFT,INTERPO,MCORLEV,MOFFSET,CARSTRCH,DIFROT_A,DIFROT_B,DIFROT_C,CALVER64"
    #keywords = "DATE,DATE__OBS,TELESCOP,INSTRUME,WAVELNTH,CAMERA,BUNIT,ORIGIN,CONTENT,QUALITY,HISTORY,COMMENT,BLD_VERS,TOTVALS_1,DATAVALS_1,MISSVALS_1,DATAMIN_1,DATAMAX_1,DATAMEDN_1,DATAMEAN_1,DATARMS_1,DATASKEW_1,DATAKURT_1,TOTVALS_2,DATAVALS_2,MISSVALS_2,DATAMIN_2,DATAMAX_2,DATAMEDN_2,DATAMEAN_2,DATARMS_2,DATASKEW_2,DATAKURT_2,TOTVALS_3,DATAVALS_3,MISSVALS_3,DATAMIN_3,DATAMAX_3,DATAMEDN_3,DATAMEAN_3,DATARMS_3,DATASKEW_3,DATAKURT_3,CTYPE1,CTYPE2,CRPIX1,CRPIX2,CRVAL1,CRVAL2,CDELT1,CDELT2,CUNIT1,CUNIT2,CROTA2,CRDER1,CRDER2,CSYSER1,CSYSER2WCSNAME,DSUN_OBS,DSUN_REF,RSUN_REF,CRLN_OBS,CRLT_OBS,HGLN_OBS,CAR_ROT,CAR_ROT2,OBS_VR,OBS_VW,OBS_VN,T_OBS,T_REC,T_REC_epoch,T_REC_step,T_REC_unit,HMI_PREV,HMI_NEXT,CADENCE,DATASIGN,MAPMMAX,MAPBMAX,MAPLGMAX,MAPLGMIN,MAPRMAX,SINBDIVS,LGSHIFT,INTERPO,MCORLEV,MOFFSET,CARSTRCH,DIFROT_A,DIFROT_B,DIFROT_C,CALVER64"
    # HISTORY keyword excluded due to the additional \n that breaks the algorithm
    
    show_info = 'show_info %s["%s"] key="%s" -iPA' 
    
    #-P for path and -A for segment
    
    si_out = subprocess.check_output(show_info %(input_ds,inRecs,keywords) , shell=True)[:-1].decode("utf-8")
    raw = si_out.split('\n')

    # Linebreak at HISTORY keyword splits the data once too many
    #raw[1] = raw[1]+raw[2]
    #raw = raw[:2]
    #print(raw)
    
    formatted = [] 
    drms_param = []

    nRecs = 0
    keys = raw[0].split('\t')

    for line in raw[1:]:  
        formatted = line.split('\t') # [CALVER64, T_REC, QUALITY, FDRADIAL, CARSTRCH, DIFROT_A, DIFROT_B, DIFROT_C, CRVAL1, CRLN_OBS, CAR_ROT, MAPLGMAX, MAPLGMIN, I_DREC]
        dict_tmp = {}
        
        for i, key in enumerate(keys):
            if formatted[i].strip() == "InvalidKeyname":
                dict_tmp[key] = 0
            else:
                dict_tmp[key] = formatted[i]
   
        drms_param.append(dict_tmp)
        nRecs += 1
    return drms_param

In [262]:
trec = "2022.03.01_00:00:00_TAI,2022.03.01_00:12:00_TAI,2022.03.01_00:24:00_TAI"
data_series = "hmi.m_720s"
path = "../output/data/crop/"

params = get_drms_parameters(trec, data_series)

In [263]:
params[1]['magnetogram']

'/SUM34/D1493189168/S00004/magnetogram.fits'

In [248]:
params[0]

{'query': 'hmi.m_720s[2022.03.01_00:00:00_TAI][3]',
 'DATE': '2022-03-05T21:27:36Z',
 'DATE__OBS': '2022-02-28T23:58:33.30Z',
 'TELESCOP': 'SDO/HMI',
 'INSTRUME': 'HMI_COMBINED',
 'WAVELNTH': '6173.0',
 'CAMERA': '3',
 'magnetogram': '/SUM34/D1493189168/S00003/magnetogram.fits'}